
# Notebook 2 — Downstream classification from saved artifacts

This notebook starts after all encoding work is done. It loads the artifacts created by Notebook 1, then runs:

1. `clf_static` on the saved static admission embeddings
2. Risk trajectory feature construction from the saved day embeddings
3. Retrieval feature loading (`wscore_top20`)
4. Final fusion and MLP training / evaluation

## Final model input used here
- static admission embedding
- risk trajectory features
- `wscore_top20`

## Why this notebook exists separately
The encoder stage is the expensive part. Splitting the workflow lets you:
- run Notebook 1 once
- save embeddings to disc
- rerun Notebook 2 as many times as you want while changing downstream settings

## What gets saved here
Inside the same `EXP_DIR`, this notebook saves into `models/`:
- `clf_static.pkl`
- scalers / imputers
- MLP checkpoints
- evaluation summaries
- prediction tables

Also saves under `models/extra_evaluation`:
- Bootstrap CIs
- Threshold sweeps
- Calibration
- Grouped CV
- Subgroup summaries

## Note: as described in README, MIMIC-III is not included in this project, authorised PhysioNet access is required.

In [ ]:

# ============================================================
# CELL 1: Paths + select experiment folder
# - Set EXPERIMENT_NAME to the exact folder created by Notebook 1
# ============================================================

from pathlib import Path

# Match this to Notebook 1
MAX_HOSP_DAY = 2
SEED = 42
EXPERIMENT_NAME = f"your experiment name"

ARTIFACT_ROOT = Path("your artifact root")
EXP_DIR = ARTIFACT_ROOT / EXPERIMENT_NAME

META_DIR = EXP_DIR / "metadata"
EMB_DIR = EXP_DIR / "embeddings"
RETR_DIR = EXP_DIR / "retrieval"
MODEL_DIR = EXP_DIR / "models"

print("Loading experiment from:")
print(EXP_DIR)

if not EXP_DIR.exists():
    raise FileNotFoundError(f"Experiment folder not found: {EXP_DIR}")


In [ ]:

# ============================================================
# CELL 2: Imports + reproducibility
# - sklearn for clf_static and preprocessing
# - torch for MLP training
# ============================================================

import gc
import json
import os
import pickle
import random
from typing import Dict, List

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("DEVICE:", DEVICE)

MODEL_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:

# ============================================================
# CELL 3: Helper functions
# - File loading
# - Evaluation
# - Risk trajectory feature builder
# - MLP training helpers
# ============================================================

def load_npz_array(path: Path) -> np.ndarray:
    return np.load(path)["array"]


def save_pickle(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(obj, f)


def evaluate_lr(name: str, clf, X: np.ndarray, y: np.ndarray) -> Dict[str, float]:
    probs = clf.predict_proba(X)[:, 1]
    out = {
        "model": name,
        "AUROC": float(roc_auc_score(y, probs)),
        "AUPRC": float(average_precision_score(y, probs)),
    }
    print(f"{name}: AUROC={out['AUROC']:.4f}  AUPRC={out['AUPRC']:.4f}")
    return out


def prob_to_logit(p: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


def compute_risk_features_for_one_admission(df_one_hadm: pd.DataFrame) -> dict:
    """
    Builds the same ComboV6-style trajectory features from day-level risk values.
    """
    daily = df_one_hadm.sort_values("hospital_day")
    x = daily["hospital_day"].values.reshape(-1, 1)
    y = daily["risk_value"].values

    slope = float(LinearRegression().fit(x, y).coef_[0]) if len(y) >= 2 else 0.0
    start = float(y[0])
    last = float(y[-1])
    delta = float(last - start)

    return {
        "risk_slope": slope,
        "risk_start": start,
        "risk_last": last,
        "risk_delta": delta,
        "risk_mean": float(np.mean(y)),
        "risk_max": float(np.max(y)),
        "risk_std": float(np.std(y)),
        "n_risk_days": int(len(y)),
    }


def build_risk_feature_table(day_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for hadm_id, g in day_df.groupby("HADM_ID"):
        feats = compute_risk_features_for_one_admission(g[["hospital_day", "risk_value"]].copy())
        feats["HADM_ID"] = int(hadm_id)
        rows.append(feats)
    return pd.DataFrame(rows)


def align_embeddings(adm_df: pd.DataFrame, X_static: np.ndarray, merged_df: pd.DataFrame) -> np.ndarray:
    """
    Aligns precomputed admission embeddings to a merged dataframe row order.
    """
    idx_map = {int(hadm): i for i, hadm in enumerate(adm_df["HADM_ID"].values)}
    rows = [idx_map[int(h)] for h in merged_df["HADM_ID"].values]
    return X_static[rows]


def make_loader(X, y, batch_size=256, shuffle=True):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.float32)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle, drop_last=False)


class SmallMLP(nn.Module):
    def __init__(self, in_dim, hidden1=256, hidden2=64, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


@torch.no_grad()
def predict_proba_nn(model, X, batch_size=4096):
    model.eval()
    probs = []
    for start in range(0, len(X), batch_size):
        xb = torch.tensor(X[start:start + batch_size], dtype=torch.float32, device=DEVICE)
        logits = model(xb)
        batch_probs = torch.sigmoid(logits).detach().cpu().numpy()
        probs.append(batch_probs)
    return np.concatenate(probs)


def evaluate_nn(name, model, X, y):
    probs = predict_proba_nn(model, X)
    out = {
        "model": name,
        "AUROC": float(roc_auc_score(y, probs)),
        "AUPRC": float(average_precision_score(y, probs)),
    }
    print(f"{name}: AUROC={out['AUROC']:.4f}  AUPRC={out['AUPRC']:.4f}")
    return out, probs


def train_mlp(
    X_train, y_train, X_val, y_val,
    hidden1=256, hidden2=64, dropout=0.3,
    lr=1e-3, weight_decay=1e-4, batch_size=256,
    max_epochs=60, patience=8, seed=42
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = SmallMLP(X_train.shape[1], hidden1, hidden2, dropout).to(DEVICE)
    train_loader = make_loader(X_train, y_train, batch_size=batch_size, shuffle=True)

    pos = float(y_train.sum())
    neg = float(len(y_train) - pos)
    pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32, device=DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_state = None
    best_val_auprc = -1.0
    bad_epochs = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optim.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()

        val_probs = predict_proba_nn(model, X_val)
        val_auprc = average_precision_score(y_val, val_probs)

        if val_auprc > best_val_auprc + 1e-4:
            best_val_auprc = val_auprc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model


In [ ]:

# ============================================================
# CELL 4: Load manifest, metadata tables, and saved embeddings
# - This is the point of the split notebook design:
#   no text re-encoding happens here
# ============================================================

with open(META_DIR / "manifest.json", "r") as f:
    manifest = json.load(f)

print("Manifest loaded:")
print(json.dumps(manifest, indent=2))

if int(manifest["max_hosp_day"]) != int(MAX_HOSP_DAY):
    raise ValueError(f"MAX_HOSP_DAY in Notebook 2 ({MAX_HOSP_DAY}) does not match manifest ({manifest['max_hosp_day']}).")

# Static admission metadata
train_adm = pd.read_parquet(META_DIR / "train_adm_meta.parquet")
val_adm   = pd.read_parquet(META_DIR / "val_adm_meta.parquet")
test_adm  = pd.read_parquet(META_DIR / "test_adm_meta.parquet")

# Static embeddings
X_train_static = load_npz_array(EMB_DIR / "static_train.npz")
X_val_static   = load_npz_array(EMB_DIR / "static_val.npz")
X_test_static  = load_npz_array(EMB_DIR / "static_test.npz")

y_train = train_adm["label"].values.astype(int)
y_val   = val_adm["label"].values.astype(int)
y_test  = test_adm["label"].values.astype(int)

print("Static embeddings loaded:")
print("X_train_static:", X_train_static.shape)
print("X_val_static:", X_val_static.shape)
print("X_test_static:", X_test_static.shape)

# Retrieval features
retrieval_features = pd.read_parquet(RETR_DIR / "retrieval_features.parquet")
print("\nRetrieval features:", retrieval_features.shape)
display(retrieval_features.head())


In [ ]:
# ============================================================
# UMAP CELL:
# UMAP of fine-tuned static admission embeddings on test set

# ============================================================
from matplotlib.lines import Line2D

# =========================
# GLOBAL FONT CONTROL
# =========================
FONT_SIZE = 17   # <-- change this to scale everything

plt.rcParams.update({
    "font.size": FONT_SIZE,
    "axes.titlesize": FONT_SIZE + 4,
    "axes.labelsize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE - 2,
    "ytick.labelsize": FONT_SIZE - 2,
    "legend.fontsize": FONT_SIZE - 3,
    "figure.titlesize": FONT_SIZE + 4
})

from umap import UMAP

# Use raw static admission embeddings from the currently loaded experiment
X_umap_input = X_test_static
y_umap = y_test.astype(int)

# Fit 2D UMAP
umap_model = UMAP(
    n_components=2,
    n_neighbors=30,
    min_dist=0.15,
    metric="cosine",
    random_state=42
)

X_umap_2d = umap_model.fit_transform(X_umap_input)

# Plot
plt.figure(figsize=(10, 6))

mask_survived = (y_umap == 0)
mask_died = (y_umap == 1)

# Plot majority class first, lighter and quieter
plt.scatter(
    X_umap_2d[mask_survived, 0],
    X_umap_2d[mask_survived, 1],
    s=8,
    alpha=0.22,
    color="steelblue",
    label="Survived",
    edgecolors="none"
)

# Plot minority class on top
plt.scatter(
    X_umap_2d[mask_died, 0],
    X_umap_2d[mask_died, 1],
    s=12,
    alpha=0.75,
    color="orange",
    label="Died",
    edgecolors="none"
)

plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.title("UMAP of fine-tuned static admission embeddings (test set)")
legend_handles = [
    Line2D([0], [0], marker='o', color='w', label='Survived',
           markerfacecolor='steelblue', markersize=8),
    Line2D([0], [0], marker='o', color='w', label='Died',
           markerfacecolor='orange', markersize=8),
]

plt.legend(handles=legend_handles, frameon=True)
plt.grid(alpha=0.2)
plt.tight_layout()
#plt.savefig(EXTRA_EVAL_DIR / "umap_static_finetuned_test.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:

# ============================================================
# CELL 5: Fit clf_static on the saved admission embeddings
# - The same clf_static is then reused to generate day-level risk features
# ============================================================

clf_static = LogisticRegression(max_iter=2000, n_jobs=-1, class_weight="balanced")
clf_static.fit(X_train_static, y_train)

lr_results = []
lr_results.append(evaluate_lr("LR static | val", clf_static, X_val_static, y_val))
lr_results.append(evaluate_lr("LR static | test", clf_static, X_test_static, y_test))

save_pickle(clf_static, MODEL_DIR / "clf_static.pkl")
print("Saved clf_static.pkl")


In [ ]:

# ============================================================
# CELL 6: Build risk trajectory features from saved day embeddings
# - Only runs when MAX_HOSP_DAY > 0
# - Uses clf_static predictions on day docs, exactly like ComboV6
# ============================================================

USE_LOGIT_FOR_FEATURES = False
LOGIT_EPS = 1e-6

if MAX_HOSP_DAY == 0:
    print("MAX_HOSP_DAY == 0 -> no day risk trajectory features will be built.")

    risk_feats_train = pd.DataFrame(columns=["HADM_ID"])
    risk_feats_val = pd.DataFrame(columns=["HADM_ID"])
    risk_feats_test = pd.DataFrame(columns=["HADM_ID"])
else:
    train_day = pd.read_parquet(META_DIR / "train_day_meta.parquet")
    val_day   = pd.read_parquet(META_DIR / "val_day_meta.parquet")
    test_day  = pd.read_parquet(META_DIR / "test_day_meta.parquet")

    X_train_day = load_npz_array(EMB_DIR / "day_train.npz")
    X_val_day   = load_npz_array(EMB_DIR / "day_val.npz")
    X_test_day  = load_npz_array(EMB_DIR / "day_test.npz")

    train_day = train_day.copy()
    val_day = val_day.copy()
    test_day = test_day.copy()

    train_day["risk_prob"] = clf_static.predict_proba(X_train_day)[:, 1]
    val_day["risk_prob"]   = clf_static.predict_proba(X_val_day)[:, 1]
    test_day["risk_prob"]  = clf_static.predict_proba(X_test_day)[:, 1]

    if USE_LOGIT_FOR_FEATURES:
        train_day["risk_value"] = prob_to_logit(train_day["risk_prob"].values, eps=LOGIT_EPS)
        val_day["risk_value"]   = prob_to_logit(val_day["risk_prob"].values, eps=LOGIT_EPS)
        test_day["risk_value"]  = prob_to_logit(test_day["risk_prob"].values, eps=LOGIT_EPS)
    else:
        train_day["risk_value"] = train_day["risk_prob"]
        val_day["risk_value"]   = val_day["risk_prob"]
        test_day["risk_value"]  = test_day["risk_prob"]

    risk_feats_train = build_risk_feature_table(train_day)
    risk_feats_val   = build_risk_feature_table(val_day)
    risk_feats_test  = build_risk_feature_table(test_day)

    print("risk_feats_train:", risk_feats_train.shape)
    print("risk_feats_val:", risk_feats_val.shape)
    print("risk_feats_test:", risk_feats_test.shape)

    risk_feats_train.to_parquet(MODEL_DIR / "risk_feats_train.parquet", index=False)
    risk_feats_val.to_parquet(MODEL_DIR / "risk_feats_val.parquet", index=False)
    risk_feats_test.to_parquet(MODEL_DIR / "risk_feats_test.parquet", index=False)


In [ ]:

# ============================================================
# CELL 7: Merge risk features + retrieval features onto admissions
# - Final retrieval input is only wscore_top20
# - Optional extra retrieval diagnostics remain saved but are not fed to the MLP
# ============================================================

# Keep the main retrieval signal only.
retr_cols_main = ["wscore_top20"]

retr_train = retrieval_features[retrieval_features["split"] == "train"][["HADM_ID"] + retr_cols_main].copy()
retr_val   = retrieval_features[retrieval_features["split"] == "val"][["HADM_ID"] + retr_cols_main].copy()
retr_test  = retrieval_features[retrieval_features["split"] == "test"][["HADM_ID"] + retr_cols_main].copy()

if MAX_HOSP_DAY == 0:
    train_fused_df = train_adm.merge(retr_train, on="HADM_ID", how="left")
    val_fused_df   = val_adm.merge(retr_val, on="HADM_ID", how="left")
    test_fused_df  = test_adm.merge(retr_test, on="HADM_ID", how="left")

    X_train_static_aligned = X_train_static.copy()
    X_val_static_aligned   = X_val_static.copy()
    X_test_static_aligned  = X_test_static.copy()
else:
    train_fused_df = (
        train_adm.merge(risk_feats_train, on="HADM_ID", how="inner")
                 .merge(retr_train, on="HADM_ID", how="left")
    )
    val_fused_df = (
        val_adm.merge(risk_feats_val, on="HADM_ID", how="inner")
               .merge(retr_val, on="HADM_ID", how="left")
    )
    test_fused_df = (
        test_adm.merge(risk_feats_test, on="HADM_ID", how="inner")
                .merge(retr_test, on="HADM_ID", how="left")
    )

    X_train_static_aligned = align_embeddings(train_adm, X_train_static, train_fused_df)
    X_val_static_aligned   = align_embeddings(val_adm, X_val_static, val_fused_df)
    X_test_static_aligned  = align_embeddings(test_adm, X_test_static, test_fused_df)

print("train_fused_df:", train_fused_df.shape)
print("val_fused_df:", val_fused_df.shape)
print("test_fused_df:", test_fused_df.shape)

display(train_fused_df.head())


In [ ]:

# ============================================================
# CELL 8: Prepare model matrices
# - Standardise static embeddings
# - Standardise risk features (if present)
# - Impute + standardise wscore_top20
# - Build ablation matrices and final fused matrices
# ============================================================

# Labels after any inner merges
y_train_f = train_fused_df["label"].values.astype(int)
y_val_f   = val_fused_df["label"].values.astype(int)
y_test_f  = test_fused_df["label"].values.astype(int)

# ---------- Static embedding block ----------
emb_scaler = StandardScaler(with_mean=True, with_std=True)
X_train_static_nn = emb_scaler.fit_transform(X_train_static_aligned)
X_val_static_nn   = emb_scaler.transform(X_val_static_aligned)
X_test_static_nn  = emb_scaler.transform(X_test_static_aligned)

# ---------- Retrieval block (single feature: wscore_top20) ----------
# We impute missing retrieval scores with the training mean.
retr_train_raw = train_fused_df[["wscore_top20"]].copy()
retr_val_raw   = val_fused_df[["wscore_top20"]].copy()
retr_test_raw  = test_fused_df[["wscore_top20"]].copy()

retr_fill_value = float(retr_train_raw["wscore_top20"].mean()) if retr_train_raw["wscore_top20"].notna().any() else 0.0

retr_train_raw = retr_train_raw.fillna(retr_fill_value)
retr_val_raw   = retr_val_raw.fillna(retr_fill_value)
retr_test_raw  = retr_test_raw.fillna(retr_fill_value)

retr_scaler = StandardScaler()
X_train_retr = retr_scaler.fit_transform(retr_train_raw.values.astype(np.float32))
X_val_retr   = retr_scaler.transform(retr_val_raw.values.astype(np.float32))
X_test_retr  = retr_scaler.transform(retr_test_raw.values.astype(np.float32))

# ---------- Risk block ----------
risk_cols = ["risk_slope","risk_start","risk_last","risk_delta","risk_mean","risk_max","risk_std","n_risk_days"]

if MAX_HOSP_DAY == 0:
    X_train_risk = None
    X_val_risk = None
    X_test_risk = None

    X_train_static_retr = np.hstack([X_train_static_nn, X_train_retr])
    X_val_static_retr   = np.hstack([X_val_static_nn, X_val_retr])
    X_test_static_retr  = np.hstack([X_test_static_nn, X_test_retr])

    X_train_final = X_train_static_retr.copy()
    X_val_final   = X_val_static_retr.copy()
    X_test_final  = X_test_static_retr.copy()
else:
    risk_scaler = StandardScaler()
    X_train_risk = risk_scaler.fit_transform(train_fused_df[risk_cols].values.astype(np.float32))
    X_val_risk   = risk_scaler.transform(val_fused_df[risk_cols].values.astype(np.float32))
    X_test_risk  = risk_scaler.transform(test_fused_df[risk_cols].values.astype(np.float32))

    X_train_static_retr = np.hstack([X_train_static_nn, X_train_retr])
    X_val_static_retr   = np.hstack([X_val_static_nn, X_val_retr])
    X_test_static_retr  = np.hstack([X_test_static_nn, X_test_retr])

    X_train_static_risk = np.hstack([X_train_static_nn, X_train_risk])
    X_val_static_risk   = np.hstack([X_val_static_nn, X_val_risk])
    X_test_static_risk  = np.hstack([X_test_static_nn, X_test_risk])

    X_train_final = np.hstack([X_train_static_nn, X_train_risk, X_train_retr])
    X_val_final   = np.hstack([X_val_static_nn, X_val_risk, X_val_retr])
    X_test_final  = np.hstack([X_test_static_nn, X_test_risk, X_test_retr])

# Save preprocessing objects for later reuse
save_pickle(emb_scaler, MODEL_DIR / "emb_scaler.pkl")
save_pickle(retr_scaler, MODEL_DIR / "retr_scaler.pkl")
save_pickle({"retr_fill_value": retr_fill_value}, MODEL_DIR / "retr_imputer.pkl")

if MAX_HOSP_DAY > 0:
    save_pickle(risk_scaler, MODEL_DIR / "risk_scaler.pkl")

print("Final train matrix shape:", X_train_final.shape)
print("Final val matrix shape:", X_val_final.shape)
print("Final test matrix shape:", X_test_final.shape)


In [ ]:

# ============================================================
# CELL 9: Train and evaluate MLP baselines + final fused model
# - MAX_HOSP_DAY == 0:
#     static only
#     static + retrieval   <-- final model
# - MAX_HOSP_DAY  > 0:
#     static only
#     static + retrieval
#     static + risk
#     static + risk + retrieval   <-- final model
# ============================================================

BATCH_SIZE = 256
HIDDEN1 = 256
HIDDEN2 = 64
DROPOUT = 0.3
LR = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 60
PATIENCE = 8

nn_results = []
prediction_tables = []

# ----------------------------
# Static-only MLP baseline
# ----------------------------
mlp_static = train_mlp(
    X_train_static_nn, y_train_f,
    X_val_static_nn, y_val_f,
    hidden1=HIDDEN1, hidden2=HIDDEN2, dropout=DROPOUT,
    lr=LR, weight_decay=WEIGHT_DECAY, batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS, patience=PATIENCE, seed=SEED
)

res_val_static, val_probs_static = evaluate_nn("MLP static | val", mlp_static, X_val_static_nn, y_val_f)
res_test_static, test_probs_static = evaluate_nn("MLP static | test", mlp_static, X_test_static_nn, y_test_f)

nn_results.extend([res_val_static, res_test_static])

save_pickle(mlp_static.state_dict(), MODEL_DIR / "mlp_static_state.pkl")

prediction_tables.append(
    val_fused_df[["HADM_ID", "SUBJECT_ID", "label"]].assign(split="val", model="mlp_static", prob=val_probs_static)
)
prediction_tables.append(
    test_fused_df[["HADM_ID", "SUBJECT_ID", "label"]].assign(split="test", model="mlp_static", prob=test_probs_static)
)

# ----------------------------
# Static + retrieval
# ----------------------------
mlp_static_retr = train_mlp(
    X_train_static_retr, y_train_f,
    X_val_static_retr, y_val_f,
    hidden1=HIDDEN1, hidden2=HIDDEN2, dropout=DROPOUT,
    lr=LR, weight_decay=WEIGHT_DECAY, batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS, patience=PATIENCE, seed=SEED
)

res_val_static_retr, val_probs_static_retr = evaluate_nn("MLP static+retrieval | val", mlp_static_retr, X_val_static_retr, y_val_f)
res_test_static_retr, test_probs_static_retr = evaluate_nn("MLP static+retrieval | test", mlp_static_retr, X_test_static_retr, y_test_f)

nn_results.extend([res_val_static_retr, res_test_static_retr])

save_pickle(mlp_static_retr.state_dict(), MODEL_DIR / "mlp_static_retr_state.pkl")

prediction_tables.append(
    val_fused_df[["HADM_ID", "SUBJECT_ID", "label"]].assign(split="val", model="mlp_static_retr", prob=val_probs_static_retr)
)
prediction_tables.append(
    test_fused_df[["HADM_ID", "SUBJECT_ID", "label"]].assign(split="test", model="mlp_static_retr", prob=test_probs_static_retr)
)

if MAX_HOSP_DAY == 0:
    final_model_name = "mlp_static_retr"
    final_val_probs = val_probs_static_retr
    final_test_probs = test_probs_static_retr
else:
    # ----------------------------
    # Static + risk
    # ----------------------------
    mlp_static_risk = train_mlp(
        X_train_static_risk, y_train_f,
        X_val_static_risk, y_val_f,
        hidden1=HIDDEN1, hidden2=HIDDEN2, dropout=DROPOUT,
        lr=LR, weight_decay=WEIGHT_DECAY, batch_size=BATCH_SIZE,
        max_epochs=MAX_EPOCHS, patience=PATIENCE, seed=SEED
    )

    res_val_static_risk, val_probs_static_risk = evaluate_nn("MLP static+risk | val", mlp_static_risk, X_val_static_risk, y_val_f)
    res_test_static_risk, test_probs_static_risk = evaluate_nn("MLP static+risk | test", mlp_static_risk, X_test_static_risk, y_test_f)

    nn_results.extend([res_val_static_risk, res_test_static_risk])

    save_pickle(mlp_static_risk.state_dict(), MODEL_DIR / "mlp_static_risk_state.pkl")

    prediction_tables.append(
        val_fused_df[["HADM_ID", "SUBJECT_ID", "label"]].assign(split="val", model="mlp_static_risk", prob=val_probs_static_risk)
    )
    prediction_tables.append(
        test_fused_df[["HADM_ID", "SUBJECT_ID", "label"]].assign(split="test", model="mlp_static_risk", prob=test_probs_static_risk)
    )

    # ----------------------------
    # Final model: static + risk + retrieval
    # ----------------------------
    mlp_final = train_mlp(
        X_train_final, y_train_f,
        X_val_final, y_val_f,
        hidden1=HIDDEN1, hidden2=HIDDEN2, dropout=DROPOUT,
        lr=LR, weight_decay=WEIGHT_DECAY, batch_size=BATCH_SIZE,
        max_epochs=MAX_EPOCHS, patience=PATIENCE, seed=SEED
    )

    res_val_final, val_probs_final = evaluate_nn("MLP static+risk+retrieval | val", mlp_final, X_val_final, y_val_f)
    res_test_final, test_probs_final = evaluate_nn("MLP static+risk+retrieval | test", mlp_final, X_test_final, y_test_f)

    nn_results.extend([res_val_final, res_test_final])

    save_pickle(mlp_final.state_dict(), MODEL_DIR / "mlp_final_state.pkl")

    prediction_tables.append(
        val_fused_df[["HADM_ID", "SUBJECT_ID", "label"]].assign(split="val", model="mlp_final", prob=val_probs_final)
    )
    prediction_tables.append(
        test_fused_df[["HADM_ID", "SUBJECT_ID", "label"]].assign(split="test", model="mlp_final", prob=test_probs_final)
    )

    final_model_name = "mlp_final"
    final_val_probs = val_probs_final
    final_test_probs = test_probs_final


In [ ]:

# ============================================================
# CELL 10: Save evaluation summaries and predictions
# - Writes clean CSV / parquet outputs for later analysis
# ============================================================

lr_results_df = pd.DataFrame(lr_results)
nn_results_df = pd.DataFrame(nn_results)
all_predictions_df = pd.concat(prediction_tables, ignore_index=True)

lr_results_df.to_csv(MODEL_DIR / "lr_results.csv", index=False)
nn_results_df.to_csv(MODEL_DIR / "nn_results.csv", index=False)
all_predictions_df.to_parquet(MODEL_DIR / "all_predictions.parquet", index=False)

summary_rows = []

# Pick out the final model's val/test rows for a compact summary
for split_name, probs, labels in [
    ("val", final_val_probs, y_val_f),
    ("test", final_test_probs, y_test_f),
]:
    summary_rows.append({
        "final_model_name": final_model_name,
        "split": split_name,
        "n": int(len(labels)),
        "prevalence": float(np.mean(labels)),
        "AUROC": float(roc_auc_score(labels, probs)),
        "AUPRC": float(average_precision_score(labels, probs)),
        "retrieval_feature_used": "wscore_top20",
        "max_hosp_day": int(MAX_HOSP_DAY),
    })

final_summary_df = pd.DataFrame(summary_rows)
final_summary_df.to_csv(MODEL_DIR / "final_summary.csv", index=False)

print("LR results:")
display(lr_results_df)

print("\nNN results:")
display(nn_results_df)

print("\nFinal summary:")
display(final_summary_df)


In [ ]:

# ============================================================
# CELL 11: Quick sanity display of the fused feature tables
# - Useful when checking that retrieval merged correctly
# - You should see wscore_top20 present in the final tables
# ============================================================

cols_to_show = ["HADM_ID", "SUBJECT_ID", "label", "wscore_top20"]
if MAX_HOSP_DAY > 0:
    cols_to_show += ["risk_slope", "risk_start", "risk_last", "risk_delta"]

print("Train fused preview:")
display(train_fused_df[cols_to_show].head())

print("Val fused preview:")
display(val_fused_df[cols_to_show].head())

print("Test fused preview:")
display(test_fused_df[cols_to_show].head())

print("\nNotebook 2 complete.")
print("Downstream models and evaluation files are saved in:")
print(MODEL_DIR)


In [ ]:
# ============================================================
# CELL 12: Extra evaluation imports + helper functions
# - Bootstrap CIs
# - Threshold sweeps
# - Calibration
# - Grouped CV
# - Subgroup summaries
# ============================================================

import gc
import math
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import StratifiedGroupKFold

EXTRA_EVAL_DIR = MODEL_DIR / "extra_evaluation"
EXTRA_EVAL_DIR.mkdir(parents=True, exist_ok=True)

print("Extra evaluation outputs will be saved to:")
print(EXTRA_EVAL_DIR)


def get_prediction_slice(pred_df: pd.DataFrame, split: str, model_name: str) -> pd.DataFrame:
    out = pred_df[(pred_df["split"] == split) & (pred_df["model"] == model_name)].copy()
    out = out.sort_values(["HADM_ID"]).reset_index(drop=True)
    return out


def bootstrap_metric_ci(y_true, y_prob, metric_fn, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    vals = []
    n = len(y_true)

    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        y_b = y_true[idx]
        p_b = y_prob[idx]

        # Skip invalid bootstrap samples that contain only one class
        if len(np.unique(y_b)) < 2:
            continue

        vals.append(metric_fn(y_b, p_b))

    vals = np.asarray(vals, dtype=float)

    return {
        "mean_boot": float(np.mean(vals)),
        "ci_low": float(np.percentile(vals, 2.5)),
        "ci_high": float(np.percentile(vals, 97.5)),
        "n_valid_boot": int(len(vals)),
    }


def threshold_sweep(y_true, y_prob, thresholds=None):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 99)

    rows = []
    for thr in thresholds:
        y_hat = (y_prob >= thr).astype(int)

        rows.append({
            "threshold": float(thr),
            "accuracy": float(accuracy_score(y_true, y_hat)),
            "balanced_accuracy": float(balanced_accuracy_score(y_true, y_hat)),
            "precision": float(precision_score(y_true, y_hat, zero_division=0)),
            "recall": float(recall_score(y_true, y_hat, zero_division=0)),
            "f1": float(f1_score(y_true, y_hat, zero_division=0)),
            "specificity": float(recall_score(y_true, y_hat, pos_label=0, zero_division=0)),
        })

    return pd.DataFrame(rows)


def choose_threshold_from_val(y_true, y_prob, objective="f1"):
    sweep_df = threshold_sweep(y_true, y_prob)

    if objective not in sweep_df.columns:
        raise ValueError(f"Objective '{objective}' not found in threshold sweep columns.")

    best_row = sweep_df.sort_values([objective, "threshold"], ascending=[False, True]).iloc[0]
    return float(best_row["threshold"]), sweep_df


def metrics_at_threshold(y_true, y_prob, threshold):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_hat = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_hat).ravel()

    out = {
        "threshold": float(threshold),
        "n": int(len(y_true)),
        "prevalence": float(np.mean(y_true)),
        "AUROC": float(roc_auc_score(y_true, y_prob)),
        "AUPRC": float(average_precision_score(y_true, y_prob)),
        "accuracy": float(accuracy_score(y_true, y_hat)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_hat)),
        "precision_PPV": float(precision_score(y_true, y_hat, zero_division=0)),
        "recall_sensitivity": float(recall_score(y_true, y_hat, zero_division=0)),
        "specificity": float(recall_score(y_true, y_hat, pos_label=0, zero_division=0)),
        "f1": float(f1_score(y_true, y_hat, zero_division=0)),
        "NPV": float(tn / max(tn + fn, 1)),
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
    }
    return out, y_hat


def expected_calibration_error(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_prob, bins) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    ece = 0.0
    rows = []

    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            continue

        conf = float(y_prob[mask].mean())
        acc = float(y_true[mask].mean())
        frac = float(mask.mean())

        ece += frac * abs(acc - conf)

        rows.append({
            "bin": int(b),
            "n_bin": int(mask.sum()),
            "mean_pred": conf,
            "frac_positive": acc,
            "abs_gap": float(abs(acc - conf)),
        })

    return float(ece), pd.DataFrame(rows)


def summarise_group_performance(df, group_col, label_col="label", prob_col="prob", min_n=20, min_pos=3):
    rows = []

    for group_name, g in df.groupby(group_col):
        row = {
            "group": str(group_name),
            "n": int(len(g)),
            "positives": int(g[label_col].sum()),
            "prevalence": float(g[label_col].mean()),
            "mean_pred": float(g[prob_col].mean()),
        }

        if len(g) >= min_n and g[label_col].sum() >= min_pos and g[label_col].nunique() == 2:
            row["AUROC"] = float(roc_auc_score(g[label_col], g[prob_col]))
            row["AUPRC"] = float(average_precision_score(g[label_col], g[prob_col]))
        else:
            row["AUROC"] = np.nan
            row["AUPRC"] = np.nan

        rows.append(row)

    return pd.DataFrame(rows).sort_values("group").reset_index(drop=True)


print("Helper functions loaded.")

In [ ]:
# ============================================================
# CELL 13: Repeated downstream seeds
# - Re-trains the downstream MLP heads with different seeds
# - Uses the already-saved frozen artifacts from Notebook 1
# ============================================================

SEEDS_TO_RUN = [7, 13, 21, 42, 84]

seed_model_inputs = {
    "mlp_static": (X_train_static_nn, y_train_f, X_val_static_nn, y_val_f, X_test_static_nn, y_test_f),
    "mlp_static_retr": (X_train_static_retr, y_train_f, X_val_static_retr, y_val_f, X_test_static_retr, y_test_f),
}

if MAX_HOSP_DAY > 0:
    seed_model_inputs["mlp_static_risk"] = (X_train_static_risk, y_train_f, X_val_static_risk, y_val_f, X_test_static_risk, y_test_f)
    seed_model_inputs["mlp_final"] = (X_train_final, y_train_f, X_val_final, y_val_f, X_test_final, y_test_f)

seed_rows = []
seed_pred_rows = []

for seed in SEEDS_TO_RUN:
    print(f"\n===== Running seed {seed} =====")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    for model_name, (Xtr, ytr, Xva, yva, Xte, yte) in seed_model_inputs.items():
        print(f"Training {model_name} @ seed={seed}")

        model = train_mlp(
            Xtr, ytr,
            Xva, yva,
            hidden1=HIDDEN1,
            hidden2=HIDDEN2,
            dropout=DROPOUT,
            lr=LR,
            weight_decay=WEIGHT_DECAY,
            batch_size=BATCH_SIZE,
            max_epochs=MAX_EPOCHS,
            patience=PATIENCE,
            seed=seed,
        )

        val_probs = predict_proba_nn(model, Xva)
        test_probs = predict_proba_nn(model, Xte)

        seed_rows.append({
            "seed": seed,
            "model": model_name,
            "split": "val",
            "AUROC": float(roc_auc_score(yva, val_probs)),
            "AUPRC": float(average_precision_score(yva, val_probs)),
        })
        seed_rows.append({
            "seed": seed,
            "model": model_name,
            "split": "test",
            "AUROC": float(roc_auc_score(yte, test_probs)),
            "AUPRC": float(average_precision_score(yte, test_probs)),
        })

        seed_pred_rows.append(
            val_fused_df[["HADM_ID", "SUBJECT_ID", "label"]].assign(
                seed=seed, split="val", model=model_name, prob=val_probs
            )
        )
        seed_pred_rows.append(
            test_fused_df[["HADM_ID", "SUBJECT_ID", "label"]].assign(
                seed=seed, split="test", model=model_name, prob=test_probs
            )
        )

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

seed_results_df = pd.DataFrame(seed_rows)
seed_predictions_df = pd.concat(seed_pred_rows, ignore_index=True)

seed_summary_df = (
    seed_results_df
    .groupby(["model", "split"], as_index=False)
    .agg(
        AUROC_mean=("AUROC", "mean"),
        AUROC_std=("AUROC", "std"),
        AUPRC_mean=("AUPRC", "mean"),
        AUPRC_std=("AUPRC", "std"),
        n_runs=("seed", "nunique"),
    )
    .sort_values(["split", "AUPRC_mean"], ascending=[True, False])
    .reset_index(drop=True)
)

seed_results_df.to_csv(EXTRA_EVAL_DIR / "seed_results_long.csv", index=False)
seed_predictions_df.to_parquet(EXTRA_EVAL_DIR / "seed_predictions.parquet", index=False)
seed_summary_df.to_csv(EXTRA_EVAL_DIR / "seed_results_summary.csv", index=False)

print("Seed-level long results:")
display(seed_results_df.head())

print("\nSeed summary (mean ± SD):")
display(seed_summary_df)

In [ ]:
# ============================================================
# CELL 14: Bootstrap confidence intervals on held-out test set
# - Uses the current run's saved all_predictions_df
# ============================================================

BOOT_N = 2000
BOOT_SEED = 42

ci_rows = []

test_models = sorted(all_predictions_df.loc[all_predictions_df["split"] == "test", "model"].unique())

for model_name in test_models:
    pred_slice = get_prediction_slice(all_predictions_df, split="test", model_name=model_name)

    y_true = pred_slice["label"].values.astype(int)
    y_prob = pred_slice["prob"].values.astype(float)

    auroc_ci = bootstrap_metric_ci(y_true, y_prob, roc_auc_score, n_boot=BOOT_N, seed=BOOT_SEED)
    auprc_ci = bootstrap_metric_ci(y_true, y_prob, average_precision_score, n_boot=BOOT_N, seed=BOOT_SEED + 1)

    ci_rows.append({
        "model": model_name,
        "n_test": int(len(y_true)),
        "prevalence": float(np.mean(y_true)),
        "AUROC_point": float(roc_auc_score(y_true, y_prob)),
        "AUROC_ci_low": auroc_ci["ci_low"],
        "AUROC_ci_high": auroc_ci["ci_high"],
        "AUPRC_point": float(average_precision_score(y_true, y_prob)),
        "AUPRC_ci_low": auprc_ci["ci_low"],
        "AUPRC_ci_high": auprc_ci["ci_high"],
        "n_valid_boot_auroc": auroc_ci["n_valid_boot"],
        "n_valid_boot_auprc": auprc_ci["n_valid_boot"],
    })

ci_results_df = pd.DataFrame(ci_rows).sort_values("AUPRC_point", ascending=False).reset_index(drop=True)
ci_results_df.to_csv(EXTRA_EVAL_DIR / "test_bootstrap_cis.csv", index=False)

print("Bootstrap 95% CIs on held-out test set:")
display(ci_results_df)

In [ ]:
# ============================================================
# CELL 15: ROC + PR curves for all test-set ablations
# ============================================================

# Force desired order
MODEL_ORDER = [
    "mlp_final",
    "mlp_static_risk",
    "mlp_static_retr",
    "mlp_static",
]

available_models = set(all_predictions_df.loc[all_predictions_df["split"] == "test", "model"].unique())
test_models = [m for m in MODEL_ORDER if m in available_models]

# Cleaner legend labels — keys must exactly match raw model names
pretty_names = {
    "mlp_final": "Final fused",
    "mlp_static_risk": "Static + risk trajectory",
    "mlp_static_retr": "Static + retrieval",
    "mlp_static": "Static only",
}

# Report colour scheme:
# steelblue, mediumpurple, indianred, orange
color_map = {
    "mlp_final": "steelblue",
    "mlp_static_risk": "mediumpurple",
    "mlp_static_retr": "indianred",
    "mlp_static": "orange",
}

# -------------------------
# ROC curves
# -------------------------
plt.figure(figsize=(10, 6))

for model_name in test_models:
    pred_slice = get_prediction_slice(all_predictions_df, split="test", model_name=model_name)
    y_true = pred_slice["label"].values.astype(int)
    y_prob = pred_slice["prob"].values.astype(float)

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = roc_auc_score(y_true, y_prob)

    display_name = pretty_names.get(model_name, model_name)
    plt.plot(
        fpr, tpr,
        color=color_map[model_name],
        linewidth=2.4,
        label=f"{display_name} (AUROC={roc_auc:.3f})"
    )

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1.5)
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves on held-out test set")
plt.legend(loc="lower right", frameon=True)
plt.grid(alpha=0.25)
plt.tight_layout()
#plt.savefig(EXTRA_EVAL_DIR / "roc_curves_test.png", dpi=300, bbox_inches="tight")
plt.show()

# -------------------------
# Precision-Recall curves
# -------------------------
plt.figure(figsize=(10, 6))
prevalence_line = None

for model_name in test_models:
    pred_slice = get_prediction_slice(all_predictions_df, split="test", model_name=model_name)
    y_true = pred_slice["label"].values.astype(int)
    y_prob = pred_slice["prob"].values.astype(float)

    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)

    display_name = pretty_names.get(model_name, model_name)
    plt.plot(
        recall, precision,
        color=color_map[model_name],
        linewidth=2.4,
        label=f"{display_name} (AUPRC={ap:.3f})"
    )

    if prevalence_line is None:
        prevalence_line = float(np.mean(y_true))

if prevalence_line is not None:
    plt.axhline(
        prevalence_line,
        linestyle="--",
        color="gray",
        linewidth=1.5,
        label=f"Prevalence={prevalence_line:.3f}"
    )

plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall curves on held-out test set")
plt.legend(loc="lower left", frameon=True)
plt.grid(alpha=0.25)
plt.tight_layout()
#plt.savefig(EXTRA_EVAL_DIR / "pr_curves_test.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# CELL 16: Threshold testing + confusion matrix for the final model
# - Picks threshold on validation set
# - Applies once to test set
# ============================================================

THRESHOLD_OBJECTIVE = "f1"   # can change to "balanced_accuracy", "precision", "recall", etc.

final_val_df = get_prediction_slice(all_predictions_df, split="val", model_name=final_model_name)
final_test_df = get_prediction_slice(all_predictions_df, split="test", model_name=final_model_name)

val_threshold, val_sweep_df = choose_threshold_from_val(
    final_val_df["label"].values,
    final_val_df["prob"].values,
    objective=THRESHOLD_OBJECTIVE,
)

test_metrics_dict, test_pred_labels = metrics_at_threshold(
    final_test_df["label"].values,
    final_test_df["prob"].values,
    threshold=val_threshold,
)

threshold_results_df = pd.DataFrame([test_metrics_dict])
threshold_results_df["threshold_objective"] = THRESHOLD_OBJECTIVE
threshold_results_df.to_csv(EXTRA_EVAL_DIR / "final_model_threshold_metrics_test.csv", index=False)

val_sweep_df.to_csv(EXTRA_EVAL_DIR / "final_model_threshold_sweep_val.csv", index=False)

print(f"Chosen validation threshold ({THRESHOLD_OBJECTIVE}): {val_threshold:.3f}")
display(threshold_results_df)

# Plot threshold sweep on validation set
plt.figure(figsize=(8, 5))
for col in ["precision", "recall", "f1", "balanced_accuracy", "specificity"]:
    plt.plot(val_sweep_df["threshold"], val_sweep_df[col], label=col)

plt.axvline(val_threshold, linestyle="--", label=f"chosen={val_threshold:.3f}")
plt.xlabel("Threshold")
plt.ylabel("Metric value")
plt.title(f"Validation threshold sweep for {final_model_name}")
plt.legend()
plt.tight_layout()
#plt.savefig(EXTRA_EVAL_DIR / "final_model_threshold_sweep_val.png", dpi=200)
plt.show()

# Confusion matrix on test set
cm = confusion_matrix(final_test_df["label"].values.astype(int), test_pred_labels)

plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.xticks([0, 1], ["Pred 0", "Pred 1"])
plt.yticks([0, 1], ["True 0", "True 1"])
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title(f"Confusion matrix on test set ({final_model_name})")

for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")

plt.tight_layout()
#plt.savefig(EXTRA_EVAL_DIR / "final_model_confusion_matrix_test.png", dpi=200)
plt.show()

print("Saved:")
print(EXTRA_EVAL_DIR / "final_model_threshold_metrics_test.csv")
print(EXTRA_EVAL_DIR / "final_model_threshold_sweep_val.csv")
print(EXTRA_EVAL_DIR / "final_model_threshold_sweep_val.png")
print(EXTRA_EVAL_DIR / "final_model_confusion_matrix_test.png")

In [ ]:
# ============================================================
# CELL 17: Calibration analysis
# - Compare static-only vs final model on test set
# ============================================================

calibration_models = ["mlp_static"]
if final_model_name not in calibration_models:
    calibration_models.append(final_model_name)

cal_rows = []

plt.figure(figsize=(6, 6))

for model_name in calibration_models:
    pred_slice = get_prediction_slice(all_predictions_df, split="test", model_name=model_name)

    y_true = pred_slice["label"].values.astype(int)
    y_prob = pred_slice["prob"].values.astype(float)

    frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy="quantile")
    ece_value, ece_bins_df = expected_calibration_error(y_true, y_prob, n_bins=10)

    cal_rows.append({
        "model": model_name,
        "n_test": int(len(y_true)),
        "prevalence": float(np.mean(y_true)),
        "Brier_score": float(brier_score_loss(y_true, y_prob)),
        "ECE_10bin": float(ece_value),
        "AUROC": float(roc_auc_score(y_true, y_prob)),
        "AUPRC": float(average_precision_score(y_true, y_prob)),
    })

    plt.plot(mean_pred, frac_pos, marker="o", label=model_name)
    ece_bins_df.to_csv(EXTRA_EVAL_DIR / f"calibration_bins_{model_name}.csv", index=False)

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed event rate")
plt.title("Calibration plot on held-out test set")
plt.legend()
plt.tight_layout()
#plt.savefig(EXTRA_EVAL_DIR / "calibration_plot_test.png", dpi=200)
plt.show()

calibration_summary_df = pd.DataFrame(cal_rows).sort_values("AUPRC", ascending=False).reset_index(drop=True)
calibration_summary_df.to_csv(EXTRA_EVAL_DIR / "calibration_summary_test.csv", index=False)

print("Calibration summary:")
display(calibration_summary_df)

In [ ]:
# ============================================================
# CELL 18: Retrieval interpretability
# - Retrieval-only discrimination on held-out test set
# - Observed mortality by wscore_top20 bin
# ============================================================

retrieval_test_df = test_fused_df[["HADM_ID", "label", "wscore_top20"]].copy()

# Reuse the same imputation value created earlier in Cell 8
retrieval_test_df["wscore_top20_filled"] = retrieval_test_df["wscore_top20"].fillna(retr_fill_value)

retrieval_only_auroc = roc_auc_score(
    retrieval_test_df["label"].values.astype(int),
    retrieval_test_df["wscore_top20_filled"].values.astype(float),
)
retrieval_only_auprc = average_precision_score(
    retrieval_test_df["label"].values.astype(int),
    retrieval_test_df["wscore_top20_filled"].values.astype(float),
)

retrieval_only_summary_df = pd.DataFrame([{
    "feature": "wscore_top20_filled",
    "n_test": int(len(retrieval_test_df)),
    "prevalence": float(retrieval_test_df["label"].mean()),
    "AUROC": float(retrieval_only_auroc),
    "AUPRC": float(retrieval_only_auprc),
    "n_missing_original": int(retrieval_test_df["wscore_top20"].isna().sum()),
    "imputation_value_from_train": float(retr_fill_value),
}])

retrieval_only_summary_df.to_csv(EXTRA_EVAL_DIR / "retrieval_only_test_summary.csv", index=False)

print("Retrieval-only test-set performance:")
display(retrieval_only_summary_df)

# -------------------------
# 7-bin quantile summary
# -------------------------
n_bins = 7

retrieval_test_df["retrieval_bin"] = pd.qcut(
    retrieval_test_df["wscore_top20_filled"],
    q=n_bins,
    duplicates="drop",
)

retrieval_bin_summary_df = (
    retrieval_test_df
    .groupby("retrieval_bin", as_index=False)
    .agg(
        n=("label", "size"),
        positives=("label", "sum"),
        mortality_rate=("label", "mean"),
        mean_wscore=("wscore_top20_filled", "mean"),
        median_wscore=("wscore_top20_filled", "median"),
    )
)

retrieval_bin_summary_df["mortality_pct"] = 100 * retrieval_bin_summary_df["mortality_rate"]

# Compact numeric labels for report figure
compact_labels = []
for interval in retrieval_bin_summary_df["retrieval_bin"]:
    compact_labels.append(f"{interval.left:.3f}\n– {interval.right:.3f}")

retrieval_bin_summary_df["bin_label"] = compact_labels
retrieval_bin_summary_df.to_csv(EXTRA_EVAL_DIR / "retrieval_bin_summary_test.csv", index=False)

# -------------------------
# Report-quality bar chart
# -------------------------
plt.figure(figsize=(10, 6))

x = np.arange(len(retrieval_bin_summary_df))
y = retrieval_bin_summary_df["mortality_pct"].values

plt.bar(
    x,
    y,
    color="steelblue",
    width=0.72
)

plt.xticks(x, retrieval_bin_summary_df["bin_label"])
plt.ylabel("Observed mortality rate (%)")
plt.xlabel("wscore_top20 bin (low to high)")
plt.title("Observed mortality across wscore_top20 bins (test set)")
plt.grid(axis="y", alpha=0.25)

# Add a little headroom
plt.ylim(0, max(y) * 1.12)

plt.tight_layout()
#plt.savefig(EXTRA_EVAL_DIR / "retrieval_mortality_by_bin_test.png", dpi=300, bbox_inches="tight")
plt.show()

print("Retrieval bin summary:")
display(retrieval_bin_summary_df)

In [ ]:
# ============================================================
# CELL 19: Grouped subject-level CV on development set only
# - Uses frozen artifacts
# - Re-fits scalers and downstream MLPs inside each fold
# - Does NOT rebuild Notebook 1 artifacts
# ============================================================

CV_N_SPLITS = 5
CV_RANDOM_STATE = 42

# You can restrict this list if runtime is too high.
CV_MODEL_NAMES = ["mlp_static", "mlp_static_retr"]
if MAX_HOSP_DAY > 0:
    CV_MODEL_NAMES += ["mlp_static_risk", "mlp_final"]

# Development set = original train + val
dev_df = pd.concat(
    [
        train_fused_df.reset_index(drop=True),
        val_fused_df.reset_index(drop=True),
    ],
    ignore_index=True,
)

y_dev = dev_df["label"].values.astype(int)
groups_dev = dev_df["SUBJECT_ID"].values.astype(int)

# Raw blocks for fold-specific re-scaling
X_dev_static_raw = np.vstack([X_train_static_aligned, X_val_static_aligned]).astype(np.float32)
X_dev_retr_raw = np.vstack([retr_train_raw.values, retr_val_raw.values]).astype(np.float32)

if MAX_HOSP_DAY > 0:
    X_dev_risk_raw = np.vstack([
        train_fused_df[risk_cols].values.astype(np.float32),
        val_fused_df[risk_cols].values.astype(np.float32),
    ])
else:
    X_dev_risk_raw = None

sgkf = StratifiedGroupKFold(n_splits=CV_N_SPLITS, shuffle=True, random_state=CV_RANDOM_STATE)

cv_rows = []

for fold_idx, (tr_idx, va_idx) in enumerate(sgkf.split(np.zeros(len(y_dev)), y_dev, groups_dev), start=1):
    print(f"\n===== CV fold {fold_idx}/{CV_N_SPLITS} =====")

    y_tr = y_dev[tr_idx]
    y_va = y_dev[va_idx]

    # ----- Static block -----
    static_scaler_cv = StandardScaler(with_mean=True, with_std=True)
    Xtr_static = static_scaler_cv.fit_transform(X_dev_static_raw[tr_idx])
    Xva_static = static_scaler_cv.transform(X_dev_static_raw[va_idx])

    # ----- Retrieval block -----
    retr_tr = X_dev_retr_raw[tr_idx].copy()
    retr_va = X_dev_retr_raw[va_idx].copy()

    retr_fill_cv = float(np.nanmean(retr_tr[:, 0])) if np.isfinite(np.nanmean(retr_tr[:, 0])) else 0.0
    retr_tr = np.nan_to_num(retr_tr, nan=retr_fill_cv)
    retr_va = np.nan_to_num(retr_va, nan=retr_fill_cv)

    retr_scaler_cv = StandardScaler()
    Xtr_retr = retr_scaler_cv.fit_transform(retr_tr.astype(np.float32))
    Xva_retr = retr_scaler_cv.transform(retr_va.astype(np.float32))

    # ----- Risk block -----
    if MAX_HOSP_DAY > 0:
        risk_scaler_cv = StandardScaler()
        Xtr_risk = risk_scaler_cv.fit_transform(X_dev_risk_raw[tr_idx])
        Xva_risk = risk_scaler_cv.transform(X_dev_risk_raw[va_idx])
    else:
        Xtr_risk = None
        Xva_risk = None

    # Build fold-specific matrices
    fold_inputs = {
        "mlp_static": (Xtr_static, Xva_static),
        "mlp_static_retr": (np.hstack([Xtr_static, Xtr_retr]), np.hstack([Xva_static, Xva_retr])),
    }

    if MAX_HOSP_DAY > 0:
        fold_inputs["mlp_static_risk"] = (np.hstack([Xtr_static, Xtr_risk]), np.hstack([Xva_static, Xva_risk]))
        fold_inputs["mlp_final"] = (np.hstack([Xtr_static, Xtr_risk, Xtr_retr]), np.hstack([Xva_static, Xva_risk, Xva_retr]))

    for model_name in CV_MODEL_NAMES:
        Xtr_fold, Xva_fold = fold_inputs[model_name]

        model = train_mlp(
            Xtr_fold, y_tr,
            Xva_fold, y_va,
            hidden1=HIDDEN1,
            hidden2=HIDDEN2,
            dropout=DROPOUT,
            lr=LR,
            weight_decay=WEIGHT_DECAY,
            batch_size=BATCH_SIZE,
            max_epochs=MAX_EPOCHS,
            patience=PATIENCE,
            seed=CV_RANDOM_STATE + fold_idx,
        )

        va_probs = predict_proba_nn(model, Xva_fold)

        cv_rows.append({
            "fold": fold_idx,
            "model": model_name,
            "n_val": int(len(y_va)),
            "prevalence_val": float(np.mean(y_va)),
            "AUROC": float(roc_auc_score(y_va, va_probs)),
            "AUPRC": float(average_precision_score(y_va, va_probs)),
        })

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

cv_results_df = pd.DataFrame(cv_rows)
cv_summary_df = (
    cv_results_df
    .groupby("model", as_index=False)
    .agg(
        AUROC_mean=("AUROC", "mean"),
        AUROC_std=("AUROC", "std"),
        AUPRC_mean=("AUPRC", "mean"),
        AUPRC_std=("AUPRC", "std"),
        n_folds=("fold", "nunique"),
    )
    .sort_values("AUPRC_mean", ascending=False)
    .reset_index(drop=True)
)

cv_results_df.to_csv(EXTRA_EVAL_DIR / "grouped_cv_results_long.csv", index=False)
cv_summary_df.to_csv(EXTRA_EVAL_DIR / "grouped_cv_results_summary.csv", index=False)

print("Grouped CV fold-by-fold results:")
display(cv_results_df)

print("\nGrouped CV summary:")
display(cv_summary_df)

In [ ]:
# ============================================================
# CALIBRATION INVESTIGATION
# ============================================================

# pull raw probs
val_df = get_prediction_slice(all_predictions_df, split="val", model_name=final_model_name)
test_df = get_prediction_slice(all_predictions_df, split="test", model_name=final_model_name)

y_val = val_df["label"].values.astype(int)
p_val = val_df["prob"].values.astype(float)

y_test = test_df["label"].values.astype(int)
p_test = test_df["prob"].values.astype(float)

# safer to use logits than raw probabilities
eps = 1e-6
logit_val = np.log(np.clip(p_val, eps, 1 - eps) / np.clip(1 - p_val, eps, 1 - eps))
logit_test = np.log(np.clip(p_test, eps, 1 - eps) / np.clip(1 - p_test, eps, 1 - eps))

# fit Platt calibrator on validation set only
platt = LogisticRegression(solver="lbfgs")
platt.fit(logit_val.reshape(-1, 1), y_val)

# calibrated probs on test
p_test_cal = platt.predict_proba(logit_test.reshape(-1, 1))[:, 1]

# compare
rows = [
    {
        "version": "raw_final",
        "AUROC": roc_auc_score(y_test, p_test),
        "AUPRC": average_precision_score(y_test, p_test),
        "Brier": brier_score_loss(y_test, p_test),
    },
    {
        "version": "platt_calibrated_final",
        "AUROC": roc_auc_score(y_test, p_test_cal),
        "AUPRC": average_precision_score(y_test, p_test_cal),
        "Brier": brier_score_loss(y_test, p_test_cal),
    },
]

cal_compare_df = pd.DataFrame(rows)
display(cal_compare_df)

In [ ]:
# ============================================================
# CALIBRATION FOLLOW-UP FOR FINAL MODEL
# - ECE: raw vs Platt
# - Calibration plot: raw vs Platt
# - Threshold selection on calibrated validation probs
# - Test confusion matrix + threshold metrics using calibrated probs
# ============================================================

# ------------------------------------------------------------
# 1) Pull raw validation/test probabilities for final model
# ------------------------------------------------------------
val_df = get_prediction_slice(all_predictions_df, split="val", model_name=final_model_name).copy()
test_df = get_prediction_slice(all_predictions_df, split="test", model_name=final_model_name).copy()

y_val = val_df["label"].values.astype(int)
p_val = val_df["prob"].values.astype(float)

y_test = test_df["label"].values.astype(int)
p_test = test_df["prob"].values.astype(float)

# ------------------------------------------------------------
# 2) Fit Platt scaling on VALIDATION ONLY
# ------------------------------------------------------------
eps = 1e-6

def to_logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

logit_val = to_logit(p_val, eps=eps)
logit_test = to_logit(p_test, eps=eps)

platt = LogisticRegression(solver="lbfgs")
platt.fit(logit_val.reshape(-1, 1), y_val)

p_val_platt = platt.predict_proba(logit_val.reshape(-1, 1))[:, 1]
p_test_platt = platt.predict_proba(logit_test.reshape(-1, 1))[:, 1]

# ------------------------------------------------------------
# 3) ECE: raw vs Platt
# ------------------------------------------------------------
ece_raw, ece_bins_raw = expected_calibration_error(y_test, p_test, n_bins=10)
ece_platt, ece_bins_platt = expected_calibration_error(y_test, p_test_platt, n_bins=10)

ece_compare_df = pd.DataFrame([
    {
        "version": "raw_final",
        "n_test": int(len(y_test)),
        "prevalence": float(np.mean(y_test)),
        "AUROC": float(roc_auc_score(y_test, p_test)),
        "AUPRC": float(average_precision_score(y_test, p_test)),
        "Brier": float(brier_score_loss(y_test, p_test)),
        "ECE_10bin": float(ece_raw),
    },
    {
        "version": "platt_calibrated_final",
        "n_test": int(len(y_test)),
        "prevalence": float(np.mean(y_test)),
        "AUROC": float(roc_auc_score(y_test, p_test_platt)),
        "AUPRC": float(average_precision_score(y_test, p_test_platt)),
        "Brier": float(brier_score_loss(y_test, p_test_platt)),
        "ECE_10bin": float(ece_platt),
    },
])

print("Raw vs Platt-calibrated test metrics:")
display(ece_compare_df)

print("\nECE bin details (raw):")
display(ece_bins_raw)

print("\nECE bin details (Platt-calibrated):")
display(ece_bins_platt)

# ------------------------------------------------------------
# 4) Calibration plot: raw vs Platt
# ------------------------------------------------------------
plt.figure(figsize=(10, 6))

frac_pos_raw, mean_pred_raw = calibration_curve(y_test, p_test, n_bins=10, strategy="quantile")
frac_pos_platt, mean_pred_platt = calibration_curve(y_test, p_test_platt, n_bins=10, strategy="quantile")

plt.plot(mean_pred_raw, frac_pos_raw, marker="o", label="Before Platt scaling", color='steelblue')
plt.plot(mean_pred_platt, frac_pos_platt, marker="o", label="After Platt scaling", color='orange')
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration", color="gray")

plt.xlabel("Mean predicted probability")
plt.ylabel("Observed event rate")
plt.title(f"Calibration comparison on held-out test set")
plt.legend()
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 5) Threshold selection using CALIBRATED validation probs
# ------------------------------------------------------------
THRESHOLD_OBJECTIVE = "f1"   # change if needed

val_thr_platt, val_sweep_platt_df = choose_threshold_from_val(
    y_val,
    p_val_platt,
    objective=THRESHOLD_OBJECTIVE,
)

# ------------------------------------------------------------
# 6) Test metrics at CALIBRATED threshold
# ------------------------------------------------------------
test_metrics_platt_dict, test_pred_labels_platt = metrics_at_threshold(
    y_test,
    p_test_platt,
    threshold=val_thr_platt,
)

threshold_platt_df = pd.DataFrame([test_metrics_platt_dict])
threshold_platt_df["threshold_objective"] = THRESHOLD_OBJECTIVE
threshold_platt_df["version"] = "platt_calibrated_final"

print(f"\nChosen calibrated validation threshold ({THRESHOLD_OBJECTIVE}): {val_thr_platt:.3f}")
print("\nThreshold metrics on test set (Platt-calibrated):")
display(threshold_platt_df)

# ------------------------------------------------------------
# 7) Plot threshold sweep for calibrated validation probs
# ------------------------------------------------------------
plt.figure(figsize=(10, 6))
for col in ["precision", "recall", "f1", "balanced_accuracy", "specificity"]:
    plt.plot(val_sweep_platt_df["threshold"], val_sweep_platt_df[col], label=col)

plt.axvline(val_thr_platt, linestyle="--", label=f"chosen={val_thr_platt:.3f}")
plt.xlabel("Threshold")
plt.ylabel("Metric value")
plt.title(f"Validation threshold sweep for Platt-calibrated {final_model_name}")
plt.legend()
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 8) Confusion matrix for calibrated test probs
# ------------------------------------------------------------

from matplotlib.colors import LinearSegmentedColormap

cm_platt = confusion_matrix(y_test, test_pred_labels_platt)

# Row-normalised values for colouring
cm_row_pct = cm_platt.astype(float) / cm_platt.sum(axis=1, keepdims=True)

# White -> steelblue colormap
steelblue_cmap = LinearSegmentedColormap.from_list(
    "white_steelblue",
    ["#ffffff", "steelblue"]
)

fig, ax = plt.subplots(figsize=(10, 6))

# Use row percentages for colour so all quadrants are meaningful
im = ax.imshow(cm_row_pct, cmap=steelblue_cmap, vmin=0, vmax=1)

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Pred 0", "Pred 1"])
ax.set_yticklabels(["True 0", "True 1"])

ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Confusion matrix on test set (Platt-calibrated MLP)")

# Annotate with count + row percentage
for i in range(2):
    for j in range(2):
        count = cm_platt[i, j]
        pct = cm_row_pct[i, j] * 100
        text_colour = "white" if cm_row_pct[i, j] > 0.5 else "black"
        ax.text(
            j, i,
            f"{count}\n({pct:.1f}%)",
            ha="center", va="center",
            color=text_colour
        )

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 9) Optional: side-by-side comparison of raw vs calibrated threshold metrics
#    using each version's OWN validation-chosen threshold
# ------------------------------------------------------------
raw_val_thr, raw_val_sweep_df = choose_threshold_from_val(
    y_val,
    p_val,
    objective=THRESHOLD_OBJECTIVE,
)

raw_test_metrics_dict, raw_test_pred_labels = metrics_at_threshold(
    y_test,
    p_test,
    threshold=raw_val_thr,
)

raw_threshold_df = pd.DataFrame([raw_test_metrics_dict])
raw_threshold_df["threshold_objective"] = THRESHOLD_OBJECTIVE
raw_threshold_df["version"] = "raw_final"

threshold_compare_df = pd.concat(
    [raw_threshold_df, threshold_platt_df],
    ignore_index=True,
)

print("\nRaw vs Platt-calibrated threshold-based test metrics:")
display(threshold_compare_df)